In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
    .appName("Module4_ChurnPrediction") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

In [5]:
import glob

folder_path = "E:/Ky2_2025_2026/BigData/CLEARDATA/*.csv"
file_list = glob.glob(folder_path)

print(f"Đã tìm thấy {len(file_list)} file CSV trong thư mục.")

if len(file_list) > 0:
    print("Đang đọc toàn bộ dữ liệu...")
    df = spark.read.csv(file_list, header=True, inferSchema=False)
    
    print("--- Cấu trúc dữ liệu ---")
    df.printSchema()
    
    print("--- 5 dòng đầu tiên ---")
    df.show(5)
else:
    print("Không tìm thấy file CSV nào, bạn kiểm tra lại đường dẫn nhé!")

Đã tìm thấy 29 file CSV trong thư mục.
Đang đọc toàn bộ dữ liệu...
--- Cấu trúc dữ liệu ---
root
 |-- user_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- recording_msid: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- release_name: string (nullable = true)

--- 5 dòng đầu tiên ---
+-------+-------------------+--------------------+--------------------+------------+--------------------+
|user_id|          timestamp|      recording_msid|          track_name| artist_name|        release_name|
+-------+-------------------+--------------------+--------------------+------------+--------------------+
|      1|2026-01-31 12:57:58|2d39940b-f575-42a...|               Auxin|        Cell|      Onwards System|
|      1|2026-01-31 13:04:37|230a1f84-609d-43d...|             Figment|        Cell|      Onwards System|
|      1|2026-01-31 13:11:07|025df4fb-5bc4-4b3...|              Geiger|        Cell|      Onw

In [9]:
from pyspark.sql.functions import col, to_timestamp, max as spark_max, count, countDistinct, datediff, lit, when

df_filtered = df.filter(col("timestamp").rlike("^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}"))
df_clean = df_filtered.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))
df_clean = df_clean.dropna(subset=["timestamp", "user_id", "recording_msid"])

latest_date_row = df_clean.select(spark_max("timestamp")).collect()[0][0]
print(f"Mốc thời gian hiện tại (dòng log mới nhất): {latest_date_row}")

user_features = df_clean.groupBy("user_id").agg(
    spark_max("timestamp").alias("last_listen_date"),   
    count("recording_msid").alias("total_listens"),         
    countDistinct("artist_name").alias("unique_artists")    
)

user_profile = user_features.withColumn(
    "days_inactive",
    datediff(lit(latest_date_row), col("last_listen_date"))
).withColumn(
    "is_churn",
    when(col("days_inactive") >= 14, 1).otherwise(0)
)

print("--- 5 dòng dữ liệu User Profile đầu tiên ---")
user_profile.show(5)

print("--- Thống kê số lượng User (Churn vs Active) ---")
user_profile.groupBy("is_churn").count().show()

<>:3: SyntaxWarning: invalid escape sequence '\d'
<>:3: SyntaxWarning: invalid escape sequence '\d'
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_18644\3139802146.py:3: SyntaxWarning: invalid escape sequence '\d'
  df_filtered = df.filter(col("timestamp").rlike("^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}"))


Mốc thời gian hiện tại (dòng log mới nhất): 2026-02-05 00:03:33
--- 5 dòng dữ liệu User Profile đầu tiên ---
+-------+-------------------+-------------+--------------+-------------+--------+
|user_id|   last_listen_date|total_listens|unique_artists|days_inactive|is_churn|
+-------+-------------------+-------------+--------------+-------------+--------+
|  19338|2026-02-04 20:22:22|         2729|           897|            1|       0|
|  28334|2026-02-04 23:58:22|         4803|          2384|            1|       0|
|  38672|2026-02-04 21:26:51|         4597|          1025|            1|       0|
|  48063|2026-02-04 23:28:06|          104|            53|            1|       0|
|  91959|2026-02-04 22:42:22|       113961|          9376|            1|       0|
+-------+-------------------+-------------+--------------+-------------+--------+
only showing top 5 rows
--- Thống kê số lượng User (Churn vs Active) ---
+--------+-----+
|is_churn|count|
+--------+-----+
|       1| 2431|
|       0|20

In [13]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import col

feature_columns = ["total_listens", "unique_artists"]

assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")

ml_data = assembler.transform(user_profile).select(
    col("user_id"), 
    col("features"), 
    col("is_churn").alias("label")
)

train_data, test_data = ml_data.randomSplit([0.8, 0.2], seed=42)
print(f"Số lượng data huấn luyện (Train): {train_data.count()}")
print(f"Số lượng data kiểm thử (Test): {test_data.count()}")

rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, maxDepth=5)

print("Đang huấn luyện mô hình Random Forest, vui lòng chờ...")
rf_model = rf.fit(train_data)
print("Huấn luyện xong!")

predictions = rf_model.transform(test_data)
print("--- Kết quả dự đoán trên tập Test (5 người đầu tiên) ---")

predictions.select("user_id", "label", "prediction", "probability").show(5, truncate=False)

evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc = evaluator.evaluate(predictions)
print(f"Độ chính xác của mô hình (AUC-ROC): {auc:.4f}")

Số lượng data huấn luyện (Train): 18818
Số lượng data kiểm thử (Test): 4558
Đang huấn luyện mô hình Random Forest, vui lòng chờ...
Huấn luyện xong!
--- Kết quả dự đoán trên tập Test (5 người đầu tiên) ---
+-------+-----+----------+----------------------------------------+
|user_id|label|prediction|probability                             |
+-------+-----+----------+----------------------------------------+
|10058  |0    |0.0       |[0.8696640327383758,0.13033596726162425]|
|1043   |0    |0.0       |[0.9834902779083389,0.01650972209166108]|
|10512  |0    |0.0       |[0.5354691931614688,0.46453080683853115]|
|10803  |0    |0.0       |[0.980573681927777,0.019426318072223038]|
|11037  |0    |0.0       |[0.7100935000176535,0.28990649998234647]|
+-------+-----+----------+----------------------------------------+
only showing top 5 rows
Độ chính xác của mô hình (AUC-ROC): 0.8815


In [17]:
import pandas as pd

print("Đang chuyển đổi dữ liệu sang Pandas để xuất file...")

pandas_df = web_data.toPandas()

print("--- 5 dòng dữ liệu chuẩn bị đưa lên Web ---")
print(pandas_df.head())

output_path = "E:/Ky2_2025_2026/BigData/web_dashboard_data.csv" 
pandas_df.to_csv(output_path, index=False)

print(f"Đã xuất file dữ liệu Web thành công tại: {output_path}")

Đang chuyển đổi dữ liệu sang Pandas để xuất file...
--- 5 dòng dữ liệu chuẩn bị đưa lên Web ---
  user_id  total_listens  unique_artists  churn_risk_percent
0   19338           2729             897                1.53
1   28334           4803            2384                1.53
2   38672           4597            1025                1.53
3   48063            104              53                9.04
4   91959         113961            9376                1.53
Đã xuất file dữ liệu Web thành công tại: E:/Ky2_2025_2026/BigData/web_dashboard_data.csv
